# M3 — train the student on a free Colab GPU

Fine-tunes one FinBERT encoder with three heads (category, materiality, direction) on the
teacher's labels, and reports fidelity against a held-out **time** period the model never saw.

This takes roughly **5 minutes on a T4** and about **3.5 hours on a laptop CPU**, which is the
whole reason for running it here.

**Before you start:** `Runtime → Change runtime type → T4 GPU`. The next cell checks it.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart session.\n"
        "Training on Colab's CPU is slower than your own laptop and not worth doing."
    )
print(torch.cuda.get_device_name(0), '| torch', torch.__version__)

## 1. Get the code

Clones the public repo. Re-run this cell after pushing changes to pick them up.

In [ ]:
import os, shutil

REPO = 'https://github.com/Ron074/distillery-news.git'
if os.path.exists('distillery-news'):
    shutil.rmtree('distillery-news')
!git clone --depth 1 $REPO
%cd distillery-news
!pip install -q 'transformers>=4.46' 'scikit-learn>=1.4' 'pandas>=2.2,<3'
print('\nready')

## 2. Upload the labels and the sample

These are **not** in git: the labels are regenerable from the teacher and the corpus is large, so
committing them would bloat the repo for no benefit. Upload both from your machine:

- `data/labels/teacher_full.jsonl` — about 5.9 MB
- `data/samples/fnspid_clean_48040_seed42.csv` — about 10.2 MB

Pick both at once in the file dialog. If the upload keeps timing out, put the two files in Google
Drive instead and use the commented-out alternative below.

In [ ]:
import os, shutil
from google.colab import files

os.makedirs('data/labels', exist_ok=True)
os.makedirs('data/samples', exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    dest = 'data/labels' if name.endswith('.jsonl') else 'data/samples'
    shutil.move(name, f'{dest}/{name}')
    print(f'  {name} -> {dest}/')

# Alternative, if uploading is painful — put the two files in a Drive folder first:
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/drive/MyDrive/distillery/teacher_full.jsonl data/labels/
# !cp /content/drive/MyDrive/distillery/fnspid_clean_48040_seed42.csv data/samples/

In [ ]:
# Sanity-check what arrived before spending GPU time on it.
import glob, json

labels = glob.glob('data/labels/*.jsonl')
samples = glob.glob('data/samples/*.csv')
assert labels, 'no .jsonl in data/labels — re-run the upload cell'
assert samples, 'no .csv in data/samples — re-run the upload cell'

n = sum(1 for _ in open(labels[0], encoding='utf-8'))
first = json.loads(open(labels[0], encoding='utf-8').readline())
print(f'labels : {labels[0]}  ({n:,} rows)')
print(f'sample : {samples[0]}')
print(f'fields : {sorted(first)}')
SAMPLE = samples[0]

## 3. Train the main model

Train on years before 2018, tune on 2018, and test on **2019 onward** — a period the model never
sees. Near-duplicate headlines are dropped before splitting so templated items cannot sit on both
sides and inflate the score through recognition rather than learning.

Class weights are applied per head. Without them, `high` materiality at 3.3% of the corpus means a
model that never predicts it still scores well — so `high` recall is printed separately rather than
being averaged away by macro-F1.

**This run gets its own tag**, so it writes `m3_3head_ckpt.json` and `preds_3head_ckpt.csv` and
never touches the committed `results/m3_3head.json`. With the default `--seed 42` it should
reproduce that file. A T4 re-run on 2026-11-12 matched every head to four decimals, with the same
best epoch. Cell 10 prints the difference, so a nonzero line means the environment changed and is
worth finding out why. The predictions file also carries each head's softmax, which M6's confidence
routing needs.

**This is not seed variance.** Same seed, same answer. To measure run-to-run spread, use a different
`--seed` (see "Next" below).

In [ ]:
!cd code && python train_student.py \
    --labels full \
    --sample ../$SAMPLE \
    --encoder yiyanghkust/finbert-pretrain \
    --epochs 3 --batch-size 32 --log-every 100 \
    --save-dir ../checkpoints/student_3head \
    --save-predictions ../results/preds_3head_ckpt.csv \
    --project stat \
    --tag 3head_ckpt

## 4. Read the result

**Fidelity is agreement with the teacher, not correctness.** Asked the same 500 randomly drawn
headlines twice, the teacher agrees with *itself* on all three fields **90.8%** of the time, so that
is the ceiling — a student at 86% is closing most of the reachable gap, not falling 14 points short
of perfect. Quote the two together.

*(An earlier draft of this cell said 93.0%. That figure came from repeats taken off the head of a
date-ordered file, so it measured how consistently the teacher reads 2009–2010 rather than the
corpus; `results/m2_summary.md` records the correction and the random-draw probe that replaced it.)*

In [ ]:
import json
import pandas as pd

run = json.load(open('results/m3_3head_ckpt.json'))
print(f"test period {run['split']['test_from']}+   n = {run['n_test']:,}")
print(f"trained on {run['n_train']:,} rows in {run['train_seconds'] / 60:.1f} min on {run['device']}\n")
for head in run['heads']:
    s = run['test'][head]
    print(f"  {head:12s} accuracy {s['accuracy']:.4f}   macro-F1 {s['macro_f1']:.4f}")

high = run['test']['materiality']['per_class']['high']
print(f"\n  high-materiality: recall {high['recall']:.3f}  precision {high['precision']:.3f}"
      f"  (n={int(high['support'])})")
# Read the ceiling from the run itself rather than retyping it: this line said 93.0%
# for weeks after the figure was superseded.
print(f"  {run['teacher_self_consistency_note'].split(';')[0]}")
print(f"  inference {run['ms_per_headline']:.2f} ms/headline on {run['device']}")

# Same rows, same config, different random state: how far apart are the two runs?
ref = json.load(open('results/m3_3head.json'))   # the committed M3 run, weights not kept
assert (run['n_train'], run['n_val'], run['n_test']) == (ref['n_train'], ref['n_val'], ref['n_test']), \
    'this run used different rows from the run on record - stop and find out why'
print('\n  this run minus the M3 run on record:')
for head in run['heads']:
    a, b = run['test'][head], ref['test'][head]
    print(f"  {head:12s} accuracy {a['accuracy'] - b['accuracy']:+.4f}   "
          f"macro-F1 {a['macro_f1'] - b['macro_f1']:+.4f}")
ref_high = ref['test']['materiality']['per_class']['high']['recall']
print(f"  high recall  {high['recall'] - ref_high:+.3f}")

# The predictions file must reproduce the reported accuracy, and its probabilities must be real ones.
p = pd.read_csv('results/preds_3head_ckpt.csv')
assert len(p) == run['n_test'] and p['uid'].is_unique, 'predictions file has the wrong rows'
for h in run['heads']:
    acc = (p[f'student_{h}'] == p[f'teacher_{h}']).mean()
    assert abs(acc - run['test'][h]['accuracy']) < 1e-6, f'{h}: file does not reproduce the reported accuracy'
    cols = [c for c in p.columns if c.startswith(f'p_{h}_')]
    assert abs(p[cols].sum(axis=1) - 1).max() < 1e-4, f'{h}: probabilities do not sum to 1'
print('\n  predictions file reproduces every reported accuracy; probabilities sum to 1')

## 5. Bring it home — the results **and** the weights

This cell used to say the weights were not worth storing: ~440 MB, git-ignored, re-run the notebook
instead. That was true when one project needed them. Two projects need them now, and four people, so
the checkpoint *is* the artifact. A re-run on a T4 with the same seed did reproduce every
reported metric to four decimals (2026-11-12). That is evidence, not a guarantee across GPU types
and library versions, and the weights are what the demo and the agent actually load.

One training run, two self-contained copies, so neither project reads the other's folder:

| copy | for |
|---|---|
| `stat_finbert_3head_<date>` | the study — fidelity, cost table, the live demo |
| `agent_finbert_3head_<date>` | the CS team's classifier tool, shared with three partners |

Both hold identical weights and the same `sha256`, and each `meta.json` records that it is one of a
pair from a single run. That is the honest record: **one model, two copies — not two models.**

Colab wipes the session when it idles, and 440 MB through `files.download` is unreliable, so the
weights go to Drive. Only the results JSON and the predictions file (a few MB) come down
through the browser.

In [ ]:
import datetime, hashlib, json, shutil
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

SRC   = Path('checkpoints/student_3head')
DEST  = Path('/content/drive/MyDrive/distillery-checkpoints')
STAMP = datetime.date.today().isoformat()

assert (SRC / 'student.pt').exists(), 'no checkpoint - run the training cell first'
assert (SRC / 'meta.json').exists(), (
    'no meta.json - this clone predates the provenance change; re-run the clone cell')

run_sha = json.loads((SRC / 'meta.json').read_text())['sha256']

for project in ('stat', 'agent'):
    out = DEST / f'{project}_finbert_3head_{STAMP}'
    if out.exists():
        shutil.rmtree(out)
    shutil.copytree(SRC, out)

    meta = json.loads((out / 'meta.json').read_text())
    meta['project'] = project
    meta['same_training_run_as'] = [p for p in ('stat', 'agent') if p != project]
    meta['copied_at'] = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
    meta['note'] = (
        'One training run, copied once per project so neither depends on the other. The weights '
        'are identical, so a figure measured on one copy holds for the other - but each project '
        'reports only what it measured itself.')
    (out / 'meta.json').write_text(json.dumps(meta, indent=2))

    # Verify the copy rather than assuming it: a truncated 440 MB write to Drive
    # looks exactly like a successful one until the model loads wrong.
    digest = hashlib.sha256()
    with open(out / 'student.pt', 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            digest.update(chunk)
    assert digest.hexdigest() == run_sha, f'{project}: copy does not match the run it claims'

    mb = (out / 'student.pt').stat().st_size / 1e6
    print(f'{project:6s} -> {out}')
    print(f'         {mb:.0f} MB   sha256 {digest.hexdigest()[:16]}...  verified')

print(f'\nboth copies match the training run {run_sha[:16]}...')
print('Drive folder:', DEST)

In [ ]:
from google.colab import files

# The run's results and its per-headline predictions with probabilities. Both belong in the repo's
# results/ folder locally; the weights are already on Drive from the cell above.
!cd results && zip -q ckpt_run.zip m3_3head_ckpt.json preds_3head_ckpt.csv && ls -l ckpt_run.zip
files.download('results/ckpt_run.zip')

## Next, once the main number looks sane

These are the remaining M3 runs. Each is the same command with one flag changed, and each takes
about the same five minutes:

```bash
# does the direction head help or hurt the other two?
python train_student.py --labels full --sample ../$SAMPLE --no-direction --tag no_direction

# learning curve: how many labels did we actually need?
for n in 1000 2500 5000 10000 20000; do
  python train_student.py --labels full --sample ../$SAMPLE --limit $n
done

# seed variance: same data and config, different random state. No --save-dir: these are
# measurements, not models to keep, and a second checkpoint folder would invite confusion
for s in 43 44; do
  python train_student.py --labels full --sample ../$SAMPLE --seed $s --tag 3head_seed$s
done

# smaller, faster student for the size comparison
python train_student.py --labels full --sample ../$SAMPLE \
    --encoder distilbert-base-uncased --tag distilbert
```

The learning curve is the one that decides whether labelling the remaining corpus would buy
anything — if the curve is flat from 10k to 20k, more labels are not the constraint.

All of these are already on record in `results/`. Re-running one here only overwrites the copy
inside this Colab session, so don't download its JSON over the committed file. Give a re-run its
own `--tag` instead.

## 6. Audit runs

Two checks that make the headline number verifiable rather than merely reported.

**Re-run with predictions saved.** The original run reported metrics about itself with no artifact
to check them against. Saving per-headline predictions lets accuracy and macro-F1 be recomputed
independently, and lets student-teacher disagreements be read rather than only counted.

**Null test.** Permute the training labels so no headline-label relationship survives, then train
at the identical config. Scores must collapse to chance. Anything better means something is
leaking the answer - and unlike every other check, this one does not depend on guessing the
failure mode in advance.

**Already done.** Both runs are on record as `results/m3_3head.json` and `results/m3_nulltest.json`.
The cells below are off by default. If re-run, they write under their own tags, because the
original tags would overwrite the committed files.


In [ ]:
# Already run; its outputs are on record. Re-running trains two more models, and under the old
# tags it would overwrite the committed files - so it is off by default and uses its own tags.
RUN_AUDIT = False

if RUN_AUDIT:
    import time
    t0 = time.time()
    # 1. the real model again, this time saving per-headline predictions
    !cd code && python train_student.py --labels full --sample ../$SAMPLE --save-predictions ../results/preds_3head_audit.csv --tag 3head_audit
    # 2. null test: identical config, training labels permuted
    !cd code && python train_student.py --labels full --sample ../$SAMPLE --shuffle-labels --tag nulltest_rerun
    print('audit runs done in', round((time.time() - t0) / 60, 1), 'min')
else:
    print('skipped: the audit is already on record (m3_3head.json, m3_nulltest.json)')


In [ ]:
# Recompute the reported metrics from the saved predictions, and read the null test beside them.
import json, pandas as pd
from sklearn.metrics import accuracy_score

if RUN_AUDIT:
    p = pd.read_csv('results/preds_3head_audit.csv')
    real = json.load(open('results/m3_3head_audit.json'))
    null = json.load(open('results/m3_nulltest_rerun.json'))

    print(f"{'head':12s} {'reported':>9s} {'recomputed':>11s} {'null':>8s}")
    for h in real['heads']:
        acc = accuracy_score(p[f'teacher_{h}'], p[f'student_{h}'])
        ok = 'OK' if abs(acc - real['test'][h]['accuracy']) < 1e-6 else 'MISMATCH'
        print(f"{h:12s} {real['test'][h]['accuracy']:>9.4f} {acc:>11.4f} "
              f"{null['test'][h]['accuracy']:>8.4f}  {ok}")

    print()
    print('The null column must sit near chance.')
    print('If it tracks the real column, stop and investigate.')


In [ ]:
from google.colab import files

if RUN_AUDIT:
    !cd results && zip -q audit.zip m3_3head_audit.json m3_nulltest_rerun.json preds_3head_audit.csv && ls -l audit.zip
    files.download('results/audit.zip')